The preliminaries

In [22]:
from openai import OpenAI
from scipy.stats import spearmanr
import jsonschema, json
import os
from dotenv import load_dotenv

length_id_batch = 25

#Hidden OpenAI API Key 
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API Key not found! Ensure you have a .env file with OPENAI_API_KEY defined.")

client = OpenAI(api_key=api_key)

#Pull in AIME and GPQA Diamond
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)

# Load the GPQA pool
with open('gpqa_pool.json', 'r') as f:
    gpqa_data = json.load(f)

#load case_ids assigned to me
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)
#print(type(aime_data))

# Load the GPQA pool
with open('asudit.json', 'r') as f:
    asudit_data = json.load(f)

# Verify the data loaded correctly
print(f"Loaded {len(aime_data)} items from AIME")
print(f"Loaded {len(gpqa_data)} items from GPQA")
print(f"Loaded {len(asudit_data)} items from asudit")

#split case_ids
gpqa_id = asudit_data["case_ids"][:length_id_batch]
aime_id = asudit_data["case_ids"][length_id_batch:]

gpqa_data_filter = {}
for id in gpqa_id:
    for case in gpqa_data:
        #print(case.keys())
        if id == case['case_id']:
            gpqa_data_filter[id] = case

aime_data_filter = {}
for id in aime_id:
    for case in aime_data:
        #print(case)
        if id == case['case_id']:
            aime_data_filter[id] = case

#print(gpqa_id)
#print(aime_id)
print(len(gpqa_data_filter), len(aime_data_filter))

Loaded 196 items from AIME
Loaded 198 items from GPQA
Loaded 4 items from asudit
25 25


1. Builds a prompt and queries the model K = 50 times with temperature=1.0 passed explicitly on every call (do not omit the parameter and rely on the API default). The exact model(s) you call are determined by your axis assignment (see the table in Your Assignment below).

In [ ]:
#Axis C mid-vs-frontier: gpt-4.1-mini (zero-shot) [Condition A] vs gpt-4.1 (zero-shot) [Condition B]


models = ["gpt-4.1-mini", "gpt-4.1"]
benchmarks = ['gpqa_diamond', 'aime']
ground_truth = ['correct_letter', 'correct_answer']


def generate_gpqa_prompt(subject):
    """
    Subject: Physics, Chemistry, or Biology
    """
    # Using an f-string to inject the 'subject' variable
    system_prompt_gpqa = f"""You are a PhD student at a top tier research university, studying {subject}.
                        Goal: Using your expert knowledge in {subject}, answer the following multiple choice questions.
                        Rules:
                        - Select one letter answer.
                        - Single-agent only. No tool-use, no multi-turn, no external retrieval.
                        Output format:
                        1. A single letter corresponding to your answer"""
    return system_prompt_gpqa

system_prompt_aime = """You are a mathematics professor at a top tier research university.
Goal: Using your expert knowledge in mathematics, solve the following math problems from the American Invitational Mathematics Examination.
Rules:
- Return a single integer answer in the range [0, 999].
- Single-agent only. No tool-use, no multi-turn, no external retrieval.
Output format:
1. A single integer in the range [0, 999] corresponding to your answer"""


def run_agent(system_prompt, model, user_message, case_id, repository, iterations=50):
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},]
    for i in range(iterations):
        resp = client.chat.completions.create(
            model=model, messages=messages, max_tokens=2000, temperature=1.0)
        repository[case_id]['answers'].append(resp.choices[0].message)
       
        
        
        #if not msg.tool_calls:
        #    return msg.content
        #for tc in msg.tool_calls:
        #    args = json.loads(tc.function.arguments)
        #    result = available_functions[tc.function.name](**args)
        #    messages.append({"role": "tool", "tool_call_id": tc.id,"content": str(result)})
gpqa_repo = {}
for id in gpqa_id:
    system_prompt_gpqa = generate_gpqa_prompt(gpqa_data_filter[id]['subject'])
    run_agent(system_prompt_gpqa, )

2. Parses each response to extract a clean answer (a single letter "A"-"D" for GPQA, a Python int in [0, 999] for AIME). Responses you cannot parse should collapse to the literal string "_UNPARSEABLE_" — the catch-all bucket for malformed model output. _UNPARSEABLE_ entries count as wrong when computing accuracy A.

3. Computes per-case metrics — consistency C, accuracy A, risk-adjusted accuracy 𝑅𝛼, and the Wilson lower bound R_wilson (formulas in The Data → Metrics below).

4. Validates and serializes the results into a single {PennKey}_results.json (for example, dkaihua_results.json) that satisfies the provided JSON schema, then computes per-cell Spearman for the report.